# Paperless-ngx ML Platform — Chameleon Provisioning

Solo build. Brings up the full integrated ML system on a single VM:

- Paperless-ngx (custom fork with HTR + semantic-search UI)
- Data stack: PostgreSQL × 2, MinIO, Redpanda, Qdrant, Adminer
- ML serving: **ml_gateway** (pretrained TrOCR + mpnet)
- **qdrant_indexer** (chunks merged_text, populates Qdrant)
- **drift_monitor** (online MMD against IAM reference)
- **htr_consumer** (Kafka→slice→HTR→DB)
- Observability: Prometheus, Grafana (pre-provisioned dashboards), Alertmanager, MLflow, rollback controller

**Pre-reqs on your local machine before running this notebook:**
- `ghcr.io/redes01/paperless-ngx-ml:latest` has been pushed to GHCR (via `scripts/build_and_push.ps1`). If you've already pushed once and haven't changed the fork, skip.
- `paperless_data` and `paperless_data_integration` repos on GitHub are up to date.

Running this notebook provisions the VM, installs Docker, clones both repos, builds every service image on the VM, and starts everything. Expect **30–45 minutes total bring-up** on first run (most of which is pip installing torch + transformers for ml_gateway and qdrant_indexer).

---
## Part 1 — VM Setup

### Step 1 — Select project and site

In [ ]:
from chi import server, context, lease, network
import chi, os, datetime, time
from datetime import timedelta, timezone

context.version = "1.0"
context.choose_project()
# CHI@TACC gives bare-metal nodes with 48+ cores and hundreds of GB of
# local disk — much more headroom than KVM@TACC's m1.xlarge (8 vCPU, 40 GB).
# Training TrOCR fine-tunes and hosting 10+ containers both want the disk.
context.choose_site(default="CHI@TACC")
username = os.getenv("USER")
print(f"Username: {username}")


### Step 2 — Reserve a bare-metal node until 2026-05-06

Node type: `compute_cascadelake` on CHI@TACC — 2× Intel Xeon Gold 6240R (48 cores total), 192 GB RAM, ~960 GB local SSD. Plenty of room for the full stack (Paperless, MinIO, Redpanda, Qdrant, MLflow, Grafana, Prometheus, ml_gateway with TrOCR + mpnet loaded, drift monitor, training runs) without disk-pressure games.

Lease runs until 2026-05-06 (~14 days). If the project's lease-length policy rejects that, shorten `lease_end` to a week and re-submit (you can extend an active lease later from the Chameleon web UI).


In [ ]:
# Lease until 2026-05-06 23:59 UTC (~14 days from today, 2026-04-22)
# If the duration fails with 'lease too long', shorten to 7 days and extend
# later via the Chameleon portal: https://chi.tacc.chameleoncloud.org/
lease_end = datetime.datetime(2026, 5, 6, 23, 59, 0, tzinfo=timezone.utc)
lease_duration = lease_end - datetime.datetime.now(timezone.utc)
print(f"Lease duration: {lease_duration}  (end: {lease_end.isoformat()})")

l = lease.Lease(
    f"lease-paperless-integration-{username}",
    duration=lease_duration,
)
# CHI bare metal: reserve by node_type, not flavor.
l.add_node_reservation(node_type="compute_cascadelake", amount=1)
# Also reserve a floating IP on CHI (on KVM, FIPs auto-allocate; on CHI
# they come from the project pool and must be requested via the lease).
l.add_fip_reservation(amount=1)
l.submit(idempotent=True)
print("Waiting for lease to become ACTIVE (can take a few minutes)...")
l.wait()
l.show()


### Step 3 — Launch the reserved node

On CHI bare metal, the `Server` uses `reservation_id` (not `flavor_name`). Provisioning takes ~5-10 min — the node physically powers on and PXE-boots the image.


In [ ]:
# Grab the node reservation id from the active lease.
# The API exposes it as l.node_reservations[0].id on python-chi >= 1.x.
# If your chi version differs, l.get_reservations() returns a list of dicts
# with an "id" field — use that as a fallback.
try:
    node_reservation_id = l.node_reservations[0].id
except Exception:
    node_reservation_id = l.get_reservations()[0]["id"]

s = server.Server(
    f"node-paperless-integration-{username}",
    reservation_id=node_reservation_id,
    image_name="CC-Ubuntu24.04",
)
s.submit(idempotent=True)
print("Node provisioning — takes 5-10 min on bare metal (PXE boot + image write)...")
s.wait()
print("Node ready.")


### Step 4 — Assign floating IP

The FIP was reserved as part of the lease in Step 2. `associate_floating_ip()` picks one from the lease's pool and attaches it.


In [ ]:
s.associate_floating_ip()
s.refresh()
s.show(type="widget")

### Step 5 — Open security groups

Ports needed:
- `22`   SSH
- `8000` Paperless UI
- `9001` MinIO console
- `5050` Adminer
- `8090` Redpanda console
- `6333` Qdrant dashboard
- `3000` Grafana
- `9090` Prometheus
- `9093` Alertmanager
- `5000` MLflow UI

In [ ]:
security_groups = [
    {"name": "allow-ssh",  "port": 22,   "description": "SSH"},
    {"name": "allow-8000", "port": 8000, "description": "Paperless UI"},
    {"name": "allow-5050", "port": 5050, "description": "Adminer"},
    {"name": "allow-9001", "port": 9001, "description": "MinIO console"},
    {"name": "allow-8090", "port": 8090, "description": "Redpanda console"},
    {"name": "allow-6333", "port": 6333, "description": "Qdrant"},
    {"name": "allow-3000", "port": 3000, "description": "Grafana"},
    {"name": "allow-9090", "port": 9090, "description": "Prometheus"},
    {"name": "allow-9093", "port": 9093, "description": "Alertmanager"},
    {"name": "allow-5000", "port": 5000, "description": "MLflow"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Applied {len(security_groups)} security groups")

### Step 6 — Install Docker

In [ ]:
s.refresh()
s.check_connectivity()
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
print("Docker installed.")

---
## Part 2 — Deploy the stack

### Step 7 — Clone repos

Both from `REDES01`. `paperless_data` contains data pipelines + quality scripts + drift reference builder; `paperless_data_integration` contains every service compose.

In [ ]:
DATA_REPO        = "https://github.com/REDES01/paperless_data.git"
INTEGRATION_REPO = "https://github.com/REDES01/paperless_data_integration.git"

s.execute("rm -rf ~/paperless_data ~/paperless_data_integration")
s.execute(f"git clone {DATA_REPO} ~/paperless_data")
s.execute(f"git clone {INTEGRATION_REPO} ~/paperless_data_integration")
s.execute("ls ~/")
print("Repos cloned.")

### Step 8 — Write Paperless secret key

Must run this before bringing up the Paperless stack.

In [ ]:
s.execute(
    "cd ~/paperless_data_integration/paperless && "
    "cp docker-compose.env.example docker-compose.env && "
    "SECRET=$(python3 -c 'import secrets; print(secrets.token_urlsafe(64))') && "
    "sed -i \"s|PAPERLESS_SECRET_KEY=replace-me-with-a-real-secret|PAPERLESS_SECRET_KEY=$SECRET|\" docker-compose.env && "
    "grep PAPERLESS_SECRET_KEY docker-compose.env"
)
print("Secret key written.")

### Step 9 — Pull Paperless image from GHCR

In [ ]:
s.execute("sg docker -c 'docker pull ghcr.io/redes01/paperless-ngx-ml:latest'")
print("Paperless image pulled.")

### Step 10 — Bring up data stack + apply ML schema migration

Starts PostgreSQL (data-stack), MinIO, Redpanda, Qdrant, Adminer. Then applies the `paperless_doc_id` migration idempotently so the HTR consumer's upserts work.

In [ ]:
s.execute("cd ~/paperless_data_integration && sg docker -c 'bash scripts/create_network.sh'")
s.execute("cd ~/paperless_data_integration && sg docker -c 'bash scripts/up_paperless_data.sh'")

print("Waiting 25s for postgres + minio to become healthy...")
time.sleep(25)

# paperless_doc_id migration (idempotent)
s.execute(
    "cat ~/paperless_data_integration/seed/phase2_add_paperless_doc_id.sql | "
    "sg docker -c 'docker exec -i postgres psql -U user -d paperless'"
)
print("Data stack up, ML schema migrated.")

### Step 11 — Bring up Paperless

In [ ]:
s.execute("cd ~/paperless_data_integration && sg docker -c 'bash scripts/up_paperless.sh'")
print("Waiting 45s for Paperless to finish starting...")
time.sleep(45)
s.execute("sg docker -c 'docker ps --filter name=paperless --format \"{{.Names}}\t{{.Status}}\"'")

### Step 12 — Create Paperless superuser + generate API token

In [ ]:
# Create superuser (admin/admin)
s.execute(
    "sg docker -c 'docker exec paperless-webserver-1 python manage.py shell -c \""
    "from django.contrib.auth.models import User; "
    "User.objects.filter(username=\\\"admin\\\").exists() or "
    "User.objects.create_superuser(\\\"admin\\\", \\\"admin@example.com\\\", \\\"admin\\\"); "
    "print(\\\"Superuser ready\\\")\"'"
)

# Fetch API token for admin (NOT User.objects.first(), which returns AnonymousUser)
result = s.execute(
    "sg docker -c 'docker exec paperless-webserver-1 python manage.py shell -c \""
    "from rest_framework.authtoken.models import Token; "
    "from django.contrib.auth.models import User; "
    "t, _ = Token.objects.get_or_create(user=User.objects.get(username=\\\"admin\\\")); "
    "print(t.key)\"'"
)
PAPERLESS_TOKEN = result.stdout.strip().split("\n")[-1]
print(f"API Token: {PAPERLESS_TOKEN}")

### Step 13 — Build and start all ML services

This step builds images on the VM for:
- `ml_gateway` (TrOCR + mpnet) — **longest, ~15 min** on first build (pip install torch + transformers + pre-download models)
- `qdrant_indexer` (mpnet cached from earlier build)
- `drift_monitor`
- `htr_consumer`
- `observability` stack (prometheus, alertmanager, grafana, mlflow, rollback_ctrl)

All containers join the shared `paperless_ml_net`. Grafana dashboards (drift + ml_gateway) are pre-provisioned from `observability/grafana/dashboards/*.json` — no manual import needed.

In [ ]:
print("Building and starting ml_gateway, qdrant_indexer, drift_monitor, observability, htr_consumer...")
print("First build is ~15 min (downloads + caches TrOCR + mpnet into ml_gateway image)")
print()

# up_all.sh orchestrates all the compose ups in dependency order
s.execute(
    f"cd ~/paperless_data_integration && "
    f"PAPERLESS_TOKEN={PAPERLESS_TOKEN} "
    f"sg docker -c 'bash scripts/up_all.sh'"
)

### Step 14 — Wait for ml_gateway to become healthy

First boot loads TrOCR + mpnet into RAM (~60s). Also give Paperless + other services time to finish warming up.

In [ ]:
print("Waiting 90s for ml_gateway startup + model load...")
time.sleep(90)

s.execute("sg docker -c 'docker ps --format \"table {{.Names}}\t{{.Status}}\"'")
print()
print("ml_gateway health check:")
s.execute("curl -sf http://localhost:8100/health || echo 'not ready yet'")

### Step 15 — Upload sample documents

Uploads the two committed samples. Paperless ingestion is async (Tesseract OCR + thumbnail); wait ~60s before looking up IDs.

In [ ]:
s.execute(
    f"for f in ~/paperless_data_integration/sample_documents/sample_budget_memo.pdf "
    f"~/paperless_data_integration/sample_documents/sample_scan.jpeg; do "
    f"echo \"Uploading $f...\"; "
    f"curl -s -X POST http://localhost:8000/api/documents/post_document/ "
    f"-H \"Authorization: Token {PAPERLESS_TOKEN}\" "
    f"-F \"document=@$f\"; "
    f"echo; done"
)
print("Uploads submitted. Waiting 60s for Paperless to ingest...")
time.sleep(60)

---
## Part 3 — Drift reference + data quality artifacts

This part produces the artifacts that the drift dashboard and training-set manifest reference in the demo video.


### Step 19 — Ingest IAM (needed by drift reference + training baseline)

This populates `s3://paperless-datalake/warehouse/iam_dataset/` with Parquet shards. Takes a couple minutes.

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install -q pyarrow minio datasets tqdm Pillow && "
    "python ingestion/ingest_iam.py\"'"
)
print("IAM ingestion complete.")

### Step 20 — Run ingestion validation (Point 1)

Produces `s3://paperless-datalake/warehouse/iam_dataset/_validation/<ts>.json`.

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install -q -r batch_pipeline/requirements.txt && "
    "python batch_pipeline/validate_ingestion.py\"'"
)

### Step 21 — Build the drift reference

Fits `MMDDriftOnline` on 500 IAM crops and saves to `s3://paperless-datalake/warehouse/drift_reference/htr_v1/cd/`. drift_monitor's background retry loop picks up the detector automatically within 60 seconds — no restart needed.

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cpu torch==2.4.1 && pip install --no-cache-dir alibi-detect pyarrow minio Pillow && "
    "python scripts/build_drift_reference.py\"'"
)


print("Reference detector uploaded to MinIO.")
print("drift_monitor's background retry loop will load it within 60 seconds.")


### Step 22 — Upload OOD samples for the drift demo

In [ ]:
s.execute(
    "cd ~/paperless_data/scripts && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "python:3.12-slim bash -c \""
    "pip install -q faker Pillow minio && "
    "python make_ood_samples.py && "
    "MINIO_ENDPOINT=minio:9000 MINIO_ACCESS_KEY=admin MINIO_SECRET_KEY=paperless_minio "
    "python upload_ood_to_minio.py\"'"
)

---
## Part 4 — Access URLs + API token

In [ ]:
s.refresh()
floating_ip = None
for net, addrs in (s.addresses or {}).items():
    for addr in addrs:
        if addr.get("OS-EXT-IPS:type") == "floating":
            floating_ip = addr["addr"]

print(f"Floating IP: {floating_ip}")
print()
print("═" * 70)
print("  Service URLs")
print("═" * 70)
print(f"  Paperless UI        http://{floating_ip}:8000           admin / admin")
print(f"  HTR review          http://{floating_ip}:8000/ml/htr-review")
print(f"  Semantic search     http://{floating_ip}:8000/ml/search")
print(f"  Grafana             http://{floating_ip}:3000           admin / admin")
print(f"  Prometheus          http://{floating_ip}:9090")
print(f"  Alertmanager        http://{floating_ip}:9093")
print(f"  MLflow              http://{floating_ip}:5000")
print(f"  MinIO Console       http://{floating_ip}:9001           admin / paperless_minio")
print(f"  Adminer             http://{floating_ip}:5050           user / paperless_postgres")
print(f"  Redpanda Console    http://{floating_ip}:8090")
print(f"  Qdrant Dashboard    http://{floating_ip}:6333/dashboard")
print()
print(f"  API Token: {PAPERLESS_TOKEN}")
print(f"  SSH:       ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")

---
## Part 5 — Training pipeline bootstrap

Seeds the MLflow `htr_training` experiment with a baseline + one fine-tuned candidate so the model registry has a deployable version before the demo. Also starts the recurring training scheduler.


### Step 23 — Build trainer image and run baseline + fine-tune (stage1)

The baseline establishes the reference CER that every later candidate's quality gate compares against. Then one fine-tune candidate runs to populate `htr/v1` in the MLflow model registry.

We use `microsoft/trocr-base-stage1` (pretrained on synthetic printed text, never fine-tuned on handwriting) as the starting point. This is the same base Microsoft used internally before fine-tuning on IAM to produce `trocr-base-handwritten` — we replicate that step on a smaller scale to demonstrate an end-to-end training pipeline that actually teaches the model something new.

On CHI `compute_cascadelake` (48 cores, CPU-only): baseline ~3 min, finetune_iam_stage1 ~30-45 min.


In [ ]:
print("Pulling latest code from git (in case of in-flight changes)...")
s.execute("cd ~/paperless_data_integration && git pull")
s.execute("cd ~/paperless_data && git pull")

print("\nBuilding trainer image (~10 min)...")
s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'docker compose -p training -f training/compose.yml build baseline_stage1'"
)

print("\nRunning baseline_stage1 (~3 min) — records reference CER into MLflow experiment tag...")
s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'docker compose -p training -f training/compose.yml run --rm baseline_stage1'"
)

print("\nRunning finetune_iam_stage1 (~30-45 min on CHI cascadelake) — should PASS the gate and register as htr/v1...")
s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'docker compose -p training -f training/compose.yml run --rm finetune_iam_stage1'"
)

print("\nVerifying registry...")
s.execute(
    "curl -s 'http://localhost:5000/api/2.0/mlflow/registered-models/get?name=htr' "
    "| python3 -m json.tool"
)


### Step 24 — Deploy the registered model to ml_gateway

Flips the `current_htr.txt` file in the shared volume to `models:/htr/1` and SIGHUPs ml_gateway. Verifies the reload via the /health endpoint.


In [ ]:
s.execute(
    "cd ~/paperless_data_integration && "
    "chmod +x scripts/deploy_model.sh && "
    "sg docker -c 'bash scripts/deploy_model.sh latest'"
)

print("\nml_gateway health after SIGHUP reload:")
s.execute("curl -s http://localhost:8100/health | python3 -m json.tool")


### Step 25 — Start recurring training scheduler

Long-running daemon that fires `finetune_combined_stage1` every `INTERVAL_MIN` minutes (default 60 for demo visibility; production would use 1440 = daily). The combined-stage1 config pulls from IAM + any user corrections saved via `/ml/htr-review`, re-fine-tunes, and registers a new `htr` version if the quality gate passes.

The scheduler skips firing if a training container is already running — safe to leave on indefinitely.


In [ ]:
s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'docker compose -p training -f training/compose.yml build training_scheduler'"
)

s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'INTERVAL_MIN=60 docker compose -p training -f training/compose.yml up -d training_scheduler'"
)

print("\nScheduler status:")
s.execute("sg docker -c 'docker ps --filter name=training_scheduler --format \"table {{.Names}}\\t{{.Status}}\"'")

print("\nFirst scheduler log lines:")
s.execute("sg docker -c 'docker logs training_scheduler 2>&1 | head -20'")


---
## Teardown

Uncomment and run when you're done to release resources.

In [ ]:
# s = server.get_server(f"node-paperless-integration-{username}")
# server.delete_server(s.id)
# l = lease.get_lease(f"lease-paperless-integration-{username}")
# lease.delete_lease(l.id)
# print("Resources released.")